# Sprint 44 — Détection de drift : comparaison des détecteurs (PC)

Galerie comparative des détecteurs de drift en ligne (S4401–S4403) évalués par le harnais
S4404 sur la grille PC S4405. On répond à **« quel détecteur, à quel coût, pour quel délai »**,
sur l'axe scientifique **supervisé ∥ non-supervisé**, et on en tire la **recommandation de
portage MCU** (Sprint 45).

> Toutes les valeurs sont **chargées** des `experiments/exp_S44_PC_*/results.json` — aucun
> chiffre en dur. RAM/latence = **proxy PC** (mesure board réelle = Sprint 45).

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.models.drift import DRIFT_DETECTORS
from src.evaluation.drift_metrics import build_comparison_table
from src.figures.catalogs import drift_detection_pc as dd

DETECTORS = [*DRIFT_DETECTORS.keys(), 'sliding_window_baseline']
DATASETS = ['synthetic', 'gas_sensor_drift', 'hydraulic', 'electricity']

def load(det, ds):
    p = ROOT / f'experiments/exp_S44_PC_{det}_{ds}/results.json'
    return json.loads(p.read_text()) if p.exists() else None

results = {det: {ds: load(det, ds) for ds in DATASETS if load(det, ds)} for det in DETECTORS}
n_cells = sum(len(v) for v in results.values())
print(f'{n_cells} cellules chargées')

36 cellules chargées


## 1. Tableau de synthèse — détecteur × dataset

Une ligne par cellule : délai, FAR, MDR, F1, MTFA/MTD, état mémoire, latence, viabilité MCU.
`None` = non calculable (honnête, ex. délai sur Electricity sans vérité-terrain ponctuelle).

In [2]:
table = build_comparison_table(results)
df = pd.DataFrame(table['rows'], columns=table['columns'])
df

,detector,dataset,requires_label,mean_detection_delay,false_alarm_rate,missed_detection_rate,f1,mtfa,mtd,state_bytes,latency_us_per_update,viabilite_mcu
0,adwin,electricity,False,NaN,0.007724,NaN,NaN,129.558739,NaN,13440,95.724043,moyenne
1,adwin,gas_sensor_drift,False,19.111111,0.029312,0.000000,0.048128,38.760563,19.111111,245760,1545.569527,pc_only
2,adwin,hydraulic,False,5.000000,0.020521,0.000000,0.097561,53.333333,5.000000,32640,175.980630,pc_only
3,adwin,synthetic,False,39.000000,0.002779,0.000000,0.285714,253.714286,39.000000,11520,79.235533,moyenne
4,ddm,electricity,True,NaN,0.003068,NaN,NaN,325.224638,NaN,20,3.572686,haute
5,ddm,gas_sensor_drift,True,138.000000,0.000659,0.777778,0.210526,1852.142857,138.000000,20,6.104394,haute
6,ddm,hydraulic,True,4.000000,0.005546,0.500000,0.153846,198.333333,4.000000,20,2.780767,haute
7,ddm,synthetic,True,57.000000,0.000000,0.333333,0.800000,NaN,57.000000,20,3.624619,haute
8,eddm,electricity,True,NaN,0.007217,NaN,NaN,137.711656,NaN,32,1.545982,haute
9,eddm,gas_sensor_drift,True,135.000000,0.000906,0.777778,0.181818,850.900000,135.000000,32,1.827691,haute


## 2. Explications prêtes à copier (par détecteur)

**Supervisés (flux d'erreur `0/1`, état O(1), label requis)**
- **DDM / EDDM** — surveillent le taux d'erreur d'un modèle de faute ; sa hausse (au-delà de
  quelques σ du minimum observé) signale le drift. État = 5–8 scalaires → **très portable MCU**.
  Coût : exigent un retour de vérité-terrain (label).
- **Page-Hinkley** — test séquentiel du cumul des écarts à la moyenne ; détection soudaine très
  réactive, état O(1).

**Non-supervisés (features/score, fenêtre bornée, autonomes)**
- **PSI / JS** — histogrammes à bacs fixes calibrés sur l'enrôlement ; état **O(bins)**
  indépendant de la fenêtre → le plus MCU-friendly des non-supervisés.
- **KSWIN / KS-Test** — test de Kolmogorov-Smirnov sur fenêtre glissante bornée O(W).
- **ADWIN** — fenêtre adaptative à buckets exponentiels (état majoré par `max_rows`).
- **MMD** — distance à noyau RBF, nativement multivariée ; stocke la référence complète
  (n_ref·d) → **coûteux en mémoire** (souvent PC-only).

**Baseline** — `SlidingWindowDriftDetector` (déjà porté C, Sprint 9/38) : fraction de la fenêtre
au-dessus d'un seuil calibré. Référence de comparaison.

## 3. Compromis délai ↔ FAR

In [3]:
fig = dd._fig_delay_vs_far()
fig

<Figure size 640x480 with 1 Axes>

En bas à gauche = idéal (rapide **et** peu de fausses alarmes). Les supervisés (rouge) réagissent
vite avec peu de fausses alarmes **mais** exigent un label ; les non-supervisés (bleu) sont
autonomes au prix d'un délai/FAR variable selon la statistique.

## 4. Coût mémoire / latence (proxy PC)

In [4]:
fig = dd._fig_cost_bars(dd.REF_DATASET)
fig

<Figure size 1100x450 with 2 Axes>

Ordre de coût attendu et vérifié : Page-Hinkley/DDM (O(1)) < PSI (O(bins)) < KSWIN/KS/ADWIN
(O(W)) < MMD (référence complète). L'annotation reporte la **viabilité MCU** dérivée de l'état
**mesuré** — la latence est un proxy PC (la mesure DWT board arrive au Sprint 45).

## 5. Déclenchement des alarmes vs vérité-terrain (synthétique)

In [5]:
fig = dd._fig_alarms_timeline('synthetic')
fig

<Figure size 900x400 with 1 Axes>

Le synthétique a des `drift_points` **exacts** `[1500, 3000, 4500]` : les alarmes alignées sur les
lignes valident la chaîne de mesure ; les traits épars ailleurs sont des fausses alarmes.

## 6. Heatmap F1 de détection + axe supervisé ∥ non-supervisé

In [6]:
fig = dd._fig_f1_heatmap()
fig

<Figure size 700x600 with 2 Axes>

In [7]:
fig = dd._fig_family_synthesis()
fig

<Figure size 640x480 with 1 Axes>

Gris = non calculable (pas de GT ponctuelle). L'axe supervisé ∥ non-supervisé résume le
compromis : **précision de détection vs coût mémoire / besoin de label (autonomie)**.

## 7. Recommandation de portage MCU (livrable pour le Sprint 45)

Classement **traçable** : chaque candidat est justifié par les chiffres mesurés ci-dessus
(état borné, latence proxy, besoin de label). La cellule suivante calcule le classement
directement depuis les `results.json` — aucun chiffre recopié à la main.

In [8]:
# Score de portabilité : état mémoire (agrégé sur les datasets) + autonomie (sans label).
def mcu_summary(det):
    cells = [r for r in results.get(det, {}).values() if r and r.get('cost')]
    if not cells:
        return None
    states = [c['cost']['state_bytes'] for c in cells if c['cost'].get('state_bytes') is not None]
    f1s = [c['drift_metrics']['f1'] for c in cells
           if c.get('drift_metrics') and c['drift_metrics'].get('f1') is not None]
    viab = [c.get('viabilite_mcu') for c in cells]
    return {
        'detector': det,
        'requires_label': cells[0].get('requires_label'),
        'state_bytes_median': int(np.median(states)) if states else None,
        'state_bytes_max': int(np.max(states)) if states else None,
        'f1_median': round(float(np.median(f1s)), 3) if f1s else None,
        'viabilite': max(set(viab), key=viab.count) if viab else None,
    }

summary = [s for det in DETECTORS if (s := mcu_summary(det))]
reco = pd.DataFrame(summary).sort_values('state_bytes_max')
reco

,detector,requires_label,state_bytes_median,state_bytes_max,f1_median,viabilite
2,page_hinkley,True,16,16,0.167,haute
0,ddm,True,20,20,0.211,haute
1,eddm,True,32,32,0.182,haute
8,sliding_window_baseline,False,200,200,0.102,haute
7,psi,False,1488,15872,0.261,moyenne
4,kswin,False,4800,51200,0.014,moyenne
5,ks_test,False,14400,153600,0.103,moyenne
3,adwin,False,23040,245760,0.098,moyenne
6,mmd,False,182126,1831424,0.100,pc_only


**Lecture (à confirmer par les mesures board S45) :**

- **Primaires** : **Page-Hinkley + DDM/EDDM** (état O(1), viabilité *haute*) et **PSI**
  (O(bins), **non-supervisé** = autonome sans label). Candidats de tête.
- **Référence** : **`SlidingWindowDriftDetector`** — déjà portée en C (`drift_detector.c`).
- **Secondaires** (à valider budget) : **ADWIN / KSWIN / KS-Test** (état O(W), viabilité *moyenne*
  à *pc_only* selon la dimensionnalité).
- **PC-only** si trop coûteux : **MMD** (stocke la référence complète `n_ref·d`).

Cette liste **entre** dans `S4501` (sélection de portage). Le résumé chiffré est repris dans
`docs/context/drift_detectors.md`.